#  Predicting Airline Flight Delays Using Machine Learning Regression

---

##  Project Overview

| Field | Details |
|---|---|
| **Project Title** | Predicting Airline Flight Delays Using Machine Learning Regression |
| **Domain** | Aviation / Transportation |
| **Task Type** | Supervised Learning — Regression |
| **Target Variable** | `ArrDelay` (Arrival Delay in minutes) |

---

##  Problem Statement

Flight delays are a persistent and costly problem in the airline industry, directly impacting passenger experience and airline operations. Several factors drive these delays — weather, air traffic congestion, carrier-side issues, and late arriving aircraft. The goal of this project is to analyze historical flight data and build a regression-based machine learning model that can predict how long a flight will be delayed (in minutes). Understanding the key factors behind delays can help airlines improve scheduling, resource allocation, and overall operational performance.

---

##  Business Objective

Build a reliable predictive model that estimates flight arrival delays using historical airline data. The model should help airlines and airport management teams make data-driven decisions — improving operational planning, reducing unnecessary delays, and enhancing the overall passenger experience through better communication and delay management.

---

## 🌻 TASK 1 — Data Acquisition & Initial Exploration

---

### Step 1 — Import Libraries

We begin by importing all the essential libraries needed throughout this project. These cover data manipulation, visualization, modeling, and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

---

### Step 2 — Load the Dataset

We load the raw flight delay dataset from a CSV file and take a quick look at the first few rows to understand its structure.

In [ ]:
df = pd.read_csv("Flight_delay.csv")
df.head(2)

In [ ]:
# Check how many records are in the full dataset
df.shape  

In [ ]:
# We work with the first 70,000 rows to keep things manageable
data = df.head(70000)

In [ ]:
# Confirm the shape of our working dataset
data.shape

In [ ]:
# List all column names
data.columns

In [ ]:
# Rearrange columns so the target variable 'ArrDelay' is at the end
cols = [col for col in data.columns if col != 'ArrDelay'] + ['ArrDelay']
data = data[cols]

In [ ]:
# Final check after reordering — target column should now be last
data.head(2)  

---

### 📊 Dataset Description

| Property | Detail |
|---|---|
| **Rows** | 70,000 |
| **Columns** | 29 |
| **Target Variable** | `ArrDelay` |
| **Prediction Task** | Regression — predicting a continuous value (delay in minutes) |

> We are dealing with a regression problem, not classification, because the target `ArrDelay` is a continuous numerical value.

In [ ]:
# Get a concise summary: column names, data types, and non-null counts
data.info()

---

## 🌻 TASK 2 — Data Preprocessing

Data preprocessing is the foundation of any machine learning project. Raw data is almost never clean — it contains missing values, duplicates, inconsistencies, and irrelevant features. We handle all of these step by step.

---

### Step 3 — Data Cleaning

#### 3.1 — Checking and Handling Missing Values

We first check which columns have missing values and how many.

In [ ]:
# Count missing values in each column
data.isnull().sum() 

In [ ]:
# Inspect the 'Org_Airport' column — checking how many unique values it has
df['Org_Airport'].value_counts()

In [ ]:
# Inspect the 'Dest_Airport' column similarly
df['Dest_Airport'].value_counts()

In [ ]:
# Both airport columns have too many unique categorical values with no clean
# encoding path — dropping them to avoid noise in the model
data.drop(['Org_Airport', 'Dest_Airport'], axis=1, inplace=True)

In [ ]:
# Confirm the shape after dropping the two airport columns
data.shape

In [ ]:
# Re-check missing values after cleanup
data.isnull().sum()

---

#### 3.2 — Handling Duplicate Rows

Duplicate records can mislead the model and inflate certain patterns in the data.

In [ ]:
# Count the number of exact duplicate rows
duplicates = data.duplicated().sum()
duplicates

In [ ]:
# Remove all duplicate rows in place
data.drop_duplicates(inplace=True)

In [ ]:
# Confirm the shape after removing duplicates
data.shape

---

#### 3.3 — Handling Data Inconsistencies and Errors

We now check for inconsistent labels, wrong data types, unnecessary columns, and any other issues that could hurt model quality.

In [ ]:
# Check data types of all columns
data.dtypes

In [ ]:
# Identify all object (text/categorical) type columns
data.select_dtypes(include='object').columns

In [ ]:
# Identify all numeric type columns
data.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
# Check the number of unique values per column — helps spot inconsistent labels
print("\nUnique Values in Each Column:")
for col in data.columns:
    print(f"{col}: {data[col].nunique()} unique")

In [ ]:
# 'TailNum' is an aircraft registration number — highly unique, not useful for modeling
data.value_counts("TailNum")  # check for every columns

In [ ]:
# Drop 'TailNum' — too many unique values, no predictive value
data.drop('TailNum', axis=1, inplace=True)

In [ ]:
# Convert 'Date' to datetime and extract Year and Month as separate features
# This allows the model to capture seasonal patterns
data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)

data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month

In [ ]:
# Drop the original 'Date' column since Year and Month now carry the needed info
data.drop('Date', axis=1, inplace=True)

**Note on `unique()` vs `value_counts()` in EDA:**

- `df.unique()` — quickly shows what categories exist; great for spotting typos or inconsistent labels.
- `df.value_counts()` — shows how often each category appears; useful for measuring class imbalance or rare values.

Best practice: inspect with `.unique()` first, then use `.value_counts()` to measure and decide how to fix any issues.

In [ ]:
# Print unique values for all categorical (object) columns to spot inconsistencies
print("\nPotential Inconsistent Categorical Values:")
cat_cols = data.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\n{col}:", data[col].unique())

In [ ]:
# Check how many unique origin airports exist in the data
data['Origin'].nunique()

In [ ]:
# 'CancellationCode' has very few unique values — mostly one dominant value
# A feature where one value dominates is not useful for a model
data.value_counts("CancellationCode")

In [ ]:
# Confirm the unique values in 'Cancelled' column
data["Cancelled"].unique()

In [ ]:
# Drop 'CancellationCode' — it provides no useful discriminating power
data.drop('CancellationCode', axis=1, inplace=True)

---

## 🌻 TASK 3 — Exploratory Data Analysis (EDA)

EDA helps us deeply understand our data before building models. We look at shape, distributions, statistical summaries, and relationships between features.

---

### Step 4 — EDA Overview

#### 4.1 — Basic Structure Inspection

We start by rechecking the shape, a sample of the data, and data types after all cleaning steps.

In [ ]:
# Check current shape of the cleaned dataset
data.shape

In [ ]:
# Quick look at the first two rows
data.head(2)

In [ ]:
# Detailed column-level info: types, non-null counts
data.info()

In [ ]:
# Data types of all columns
data.dtypes

#### 4.2 — Missing Values & Duplicates

Already handled in the preprocessing step above. No further action needed here.

In [ ]:
# Statistical summary: mean, std, min, max, percentiles for all numeric features
data.describe()

---

### Step 5 — Feature Encoding

Before we can handle outliers or build any model, we need to convert categorical columns into numbers. The machine learning algorithms we use can only work with numeric data.

We need to encode first because our outlier detection (IQR) and scaling methods all require numeric input.

In [ ]:
# Check which columns are still text/categorical
data.select_dtypes(include="object").columns

In [ ]:
# How many airlines are in the 'UniqueCarrier' column?
data['UniqueCarrier'].value_counts()

In [ ]:
# What are the exact carrier codes?
data['UniqueCarrier'].unique()

In [ ]:
# Check shape before encoding
data.shape

In [ ]:
# Apply One-Hot Encoding to 'UniqueCarrier'
# drop_first=True removes one dummy to avoid multicollinearity
data = pd.get_dummies(data, columns=['UniqueCarrier'], drop_first=True)

In [ ]:
# Shape after encoding 'UniqueCarrier'
data.shape

In [ ]:
# Inspect the 'Airline' column
data['Airline'].value_counts()

In [ ]:
# Apply One-Hot Encoding to 'Airline' as well
data = pd.get_dummies(data, columns=['Airline'], drop_first=True)

In [ ]:
# Check 'Origin' — how many categories?
data['Origin'].value_counts()

In [ ]:
# 'Origin' has a large number of unique airport codes
data['Origin'].nunique()

In [ ]:
# Because Origin has too many unique values, One-Hot Encoding would explode the
# feature space. Instead, we use Frequency Encoding — replace each airport code
# with how often it appears in the data.
freq = data['Origin'].value_counts()
data['Origin'] = data['Origin'].map(freq)

In [ ]:
# Same inspection for destination airports
data['Dest'].value_counts()

In [ ]:
# Confirm the number of unique destination airports
data['Dest'].nunique()

In [ ]:
# Apply Frequency Encoding to 'Dest' as well
freq = data['Dest'].value_counts()
data['Dest'] = data['Dest'].map(freq)

In [ ]:
# Confirm all dtypes are now numeric — no more object columns
data.info()

In [ ]:
# Check if any boolean columns were created by get_dummies
data.select_dtypes(include="bool").columns

In [ ]:
# Inspect one of the boolean dummy columns to understand the values
data['Airline_Delta Air Lines Inc.'].value_counts()

In [ ]:
# Convert all boolean columns to integer (0/1) for compatibility with sklearn
bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)

In [ ]:
# Final check — all columns should now be numeric
data.info()

In [ ]:
# Updated statistical summary after encoding
data.describe()

---

### Step 6 — Distribution Check (Skewness Analysis)

Before handling outliers, it is important to understand the distribution of our data. We use `DayOfWeek` as an example to demonstrate how to check for skewness and data spread.

In [ ]:
# Skewness of DayOfWeek — tells us how symmetric the distribution is
data['DayOfWeek'].skew()

**Skewness Interpretation:**

| Skew Range | Distribution Nature |
|---|---|
| -0.5 to 0.5 | Approximately Normal |
| 0.5 to 1.0 | Moderately Skewed |
| Above 1.0 | Highly Skewed |

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Visualize the distribution of DayOfWeek using a histogram with KDE
sns.histplot(data['DayOfWeek'], kde=True)
plt.show()

**Coefficient of Variation (CV) Analysis:**

From the statistical summary:
- Mean = 3.85
- STD = 1.92

We calculate CV to understand the spread relative to the mean.

In [ ]:
# CV = (Standard Deviation / Mean) * 100
1.92/3.85 *100

**CV Interpretation:**

| CV Value | Meaning |
|---|---|
| Less than 20% | Data is very stable, low spread |
| 20% – 40% | Moderate variation |
| Above 40% | High variation / high spread |

**Another way to check skewness — Mean vs Median comparison:**

In a perfectly normal distribution: **Mean ≈ Median ≈ Mode**

In our `DayOfWeek` data:
- Mean = 3.5
- Median = 4.0

This tells us the data is slightly **left skewed** — the distribution leans a bit to the left.

---

### Step 7 — Outlier Detection and Handling

#### 3.5 — Handling Outliers

Outliers are extreme values that don't fit the general pattern of the data. They can distort model training and lead to poor predictions. We use the **IQR (Interquartile Range)** method to detect and handle them.

In [ ]:
# Calculate outlier percentage for every feature using the IQR method
def calculate_outlier_percentages(data):
    outlier_percentages = {}
    for col in data.columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)][col]
        outlier_percentages[col] = len(outliers) / len(data) * 100  # Percentage of outliers
    return outlier_percentages

outlier_percentages_all = calculate_outlier_percentages(data)
outlier_percentages_all

**Types of outliers we commonly encounter:**

**In Numeric Data:**
Extreme high/low values, impossible values, data entry errors, unit problems, and skewness-based extremes.
Examples: age, salary, house price, weight, temperature, height, transaction amounts, exam marks.

**In Categorical Data:**
Rare categories, typos, inconsistent labels, unknown categories, encoding problems, extra spaces, case sensitivity issues.
Examples: Gender, City, Department, Country.

**Our decision rule:** We treat any feature with outlier percentage above **10%** as a priority for cleaning.

Features requiring attention: `NASDelay`, `Airline_Skywest Airlines Inc.`, `Airline_United Air Lines Inc.`

In [ ]:
# Check the distribution of values in 'NASDelay'
data["NASDelay"].value_counts()

In [ ]:
# Visualize 'NASDelay' using a boxplot to see the outlier spread
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(15,5))
sns.boxplot(x= "NASDelay",data= data, color='lightblue')
plt.show()

In [ ]:
# Distribution plot of 'NASDelay' — see the shape before removing outliers
sns.distplot(data["NASDelay"])
plt.show()

In [ ]:
# Define a reusable function to compute IQR-based outlier boundaries
def iqr_outlier_range(data):
    Q1 = np.percentile(data, 25)  # First quartile (25th percentile)
    Q3 = np.percentile(data, 75)  # Third quartile (75th percentile)
    IQR = Q3 - Q1                 # Interquartile range
    # Define outlier boundaries using the standard 1.5x IQR rule
    min_range = Q1 - 1.5 * IQR
    max_range = Q3 + 1.5 * IQR

    print(f"Minimum Outlier Range: {min_range}")
    print(f"Maximum Outlier Range: {max_range}")
    return min_range, max_range

In [ ]:
# Apply the function to compute IQR boundaries for 'NASDelay'
min_range, max_range = iqr_outlier_range(data['NASDelay'])

In [ ]:
# Remove the outlier rows — keep only values within the IQR bounds
data = data[(data['NASDelay'] >= min_range) & (data['NASDelay'] <= max_range)]

In [ ]:
# Distribution plot after outlier removal — distribution should look cleaner now
sns.distplot(data["NASDelay"])
plt.show()

In [ ]:
# Recalculate outlier percentages across all features after cleaning 'NASDelay'
def calculate_outlier_percentages(data):
    outlier_percentages = {}
    for col in data.columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)][col]
        outlier_percentages[col] = len(outliers) / len(data) * 100
    return outlier_percentages

outlier_percentages_all = calculate_outlier_percentages(data)
outlier_percentages_all

**Result:** Before cleaning, `NASDelay` had **12.68%** outliers. After applying IQR-based removal, it came down to **8.48%** — now within an acceptable range.

---

## 🌻 TASK 4 — Feature Selection

Feature selection is one of the trickiest parts of a machine learning project. The right technique depends on both the **type of target** (continuous vs categorical) and the **nature of the relationship** (linear vs non-linear) between features and the target.

**Getting this wrong leads to choosing the wrong features — which directly hurts model performance.**

### Which Technique to Use?

**For a Continuous Target (Regression like ours):**
1. **Pearson Correlation** — captures linear relationships
2. **Mutual Information Regression** — captures both linear and non-linear relationships
3. **Random Forest Importance** — acts as confirmation and handles non-linearity naturally

**For a Categorical Target (Classification):**
1. Chi-Square (for categorical features) / ANOVA (for continuous features)
2. Mutual Information Classification
3. Random Forest Importance

> **If I had to pick one technique without seeing the data first, I would choose Mutual Information — because it handles both linear and non-linear relationships.**

---

### Final Decision for This Project

Since we are using tree-based models (Random Forest, Gradient Boosting), they automatically handle feature selection during training using an **Embedded Method**. Tree-based models capture both linear and non-linear relationships inherently.

This reduces our dependency on traditional filter methods like Pearson correlation alone. That said, filter methods are still useful for exploratory analysis and understanding the data before modeling.

In [ ]:
import pandas as pd
import numpy as np

# Correlation threshold — features correlated above this are considered redundant
threshold = 0.90

# Build the absolute correlation matrix
corr_matrix = data.corr(numeric_only=True).abs()

# Look at the upper triangle only (to avoid counting each pair twice)
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find and store all feature pairs with correlation above the threshold
high_corr_pairs = []

for col in upper.columns:
    for row in upper.index:
        corr_value = upper.loc[row, col]

        if pd.notna(corr_value) and corr_value > threshold:
            high_corr_pairs.append(
                [row, col, round(corr_value, 4)]
            )

# Convert results to a readable DataFrame
high_corr_df = pd.DataFrame(
    high_corr_pairs,
    columns=[
        "Feature_1",
        "Feature_2",
        "Correlation"
    ]
)

# Sort by strongest correlation at the top
high_corr_df = high_corr_df.sort_values(
    by="Correlation",
    ascending=False
)

print(high_corr_df)

**Note on Pearson Correlation:**

Pearson Correlation only captures **linear relationships** between features. So even if two features are strongly related in a non-linear way, Pearson may not catch it.

Also important: when we are checking **feature vs feature redundancy** (not feature vs target), the concern about linearity vs non-linearity applies less — we are just looking for duplicate/redundant signals.

The main limitation is when using Pearson to measure feature importance for the target — that is where Mutual Information or tree-based methods are more reliable.

---

## 🌻 TASK 5 — Feature Scaling

Before training, we need to scale our features. Scaling ensures that no single feature dominates the model just because it has a larger numerical range. We use **StandardScaler** which transforms each feature to have mean = 0 and standard deviation = 1.

In [ ]:
# Quick look at the data before separating features and target
data.head(2)

In [ ]:
# Separate features (X) and target (y)
x = data.drop(columns='ArrDelay')
y = data['ArrDelay']

In [ ]:
# Confirm feature matrix shape
x.shape

In [ ]:
# Confirm target vector shape
y.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fit and transform the feature matrix
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [ ]:
# Convert the scaled numpy array back to a DataFrame
# This is important because correlation and feature importance methods need a DataFrame
x_final = pd.DataFrame(x_scaled, columns=data.drop(columns=['ArrDelay']).columns)

In [ ]:
# Preview the scaled features
x_final.head(2)

---

## 🌻 TASK 6 — Model Selection and Training

We compare multiple regression models to find the best-performing one. The models we evaluate are: Linear Regression, Decision Tree, Random Forest, Gradient Boosting, and Extra Trees.

**Evaluation Metrics:**
- **MAE (Mean Absolute Error)** — average absolute difference between predicted and actual values
- **RMSE (Root Mean Squared Error)** — penalizes larger errors more heavily
- **R² Score** — how much variance in the target the model explains (closer to 1 is better)

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Split the data: 80% training, 20% testing
x_train, x_test, y_train, y_test = train_test_split(
    x_final, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Define all models we want to compare
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42)
}

# Train and evaluate each model
for name, model in models.items():

    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    print(f"\n{name}")
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R2   : {r2:.3f}")

In [ ]:
# View all feature names after scaling
x.columns.tolist()

---

## 🌻 TASK 7 — Model Optimization

### Step 1 — Embedded Feature Selection (Random Forest Importance)

Rather than using a separate filter method, we let the Random Forest model itself tell us which features matter most. This is called an **Embedded Method** — feature selection happens during training.

We rank features by their importance scores and select the top ones for the final model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train a Random Forest on the full (unscaled) feature set to get importance scores
model = RandomForestRegressor(random_state=42)
model.fit(x, y)

In [ ]:
# Build a DataFrame of feature names and their importance scores, sorted descending
feature_importance = pd.DataFrame({
    'Feature': x.columns,
    'Importance': model.feature_importances_
})

feature_importance = feature_importance.sort_values(by='Importance', ascending=False)
feature_importance

In [ ]:
# Visualize the top 15 most important features using a horizontal bar chart
top_features = feature_importance.head(15)

plt.figure(figsize=(10,6))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.gca().invert_yaxis()
plt.title("Top 15 Important Features (Embedded Method)")
plt.xlabel("Importance Score")
plt.show()

In [ ]:
# Select the top 20 features based on importance scores
top_k = 20   # you can adjust this number based on your requirements

selected_features = feature_importance.head(top_k)['Feature']

# Create a reduced feature matrix with only the selected features
X_selected = x[selected_features] 

In [ ]:
# Preview the reduced feature set
X_selected.head()

In [ ]:
# Confirm which features were selected and the total count
print("Selected Features Count:", len(selected_features))
print(selected_features.tolist())

---

### Step 2 — Hyperparameter Tuning with GridSearchCV + Cross Validation

**Why use GridSearchCV instead of tuning and cross-validating separately?**

GridSearchCV combines both steps into one powerful workflow:
- It systematically tries every combination of hyperparameters we provide
- For each combination, it runs k-fold cross-validation to evaluate performance
- This prevents overfitting and gives us a reliable estimate of how well each configuration will generalize

This is much better than tuning and then cross-validating separately.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold

# 1. Initialize the base model
rf = RandomForestRegressor(random_state=42)

# 2. Define the hyperparameter search space
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# 3. Set up 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 4. Set up the grid search with cross-validation
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_absolute_error',  # We minimize MAE
    n_jobs=-1,                          # Use all CPU cores for speed
    verbose=1
)

# 5. Run the grid search
grid_search.fit(x_train, y_train)

# 6. Print the best hyperparameters and corresponding CV score
print("Best Hyperparameters:", grid_search.best_params_)
print("Best CV MAE:", -grid_search.best_score_)

# 7. Extract the best trained model
best_model = grid_search.best_estimator_

---

### Step 3 — Train the Final Optimized Model

Using the best hyperparameters found by GridSearchCV, we train our final Random Forest model and evaluate it on the held-out test set.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Build the final model with the best hyperparameters from GridSearch
rf_best = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

# Train on the training set
rf_best.fit(x_train, y_train)

# Generate predictions on the test set
y_pred = rf_best.predict(x_test)

In [ ]:
# Evaluate the final model performance
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Random Forest Final Model Results:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R2   :", r2)

---

## ✅ Project Summary

| Stage | What We Did |
|---|---|
| Data Loading | Loaded 70,000 records from Flight_delay.csv |
| Data Cleaning | Removed nulls, dropped irrelevant columns, fixed data types |
| Duplicate Handling | Removed all duplicate rows |
| Encoding | One-Hot Encoding for low-cardinality categoricals; Frequency Encoding for high-cardinality ones |
| Outlier Handling | IQR-based removal for features with >10% outlier rate |
| Feature Selection | Embedded method using Random Forest importance scores |
| Scaling | StandardScaler applied to all features |
| Model Comparison | Linear Regression, Decision Tree, Random Forest, Gradient Boosting, Extra Trees |
| Hyperparameter Tuning | GridSearchCV with 5-fold cross-validation |
| Final Model | Optimized Random Forest Regressor |

> The final model is evaluated using MAE, RMSE, and R² Score — giving us a complete picture of both average error and overall predictive power.